# Trazo en Kaggle

Servidor de **trazo** (chat, Qwen 2.5 Coder 7B + Modelfile de UML Studio) y **qwen3-vl:8b-instruct** (lectura de imágenes) con Ollama en la GPU de Kaggle, publicado con un túnel de Cloudflare.

**Antes de empezar** (panel derecho del notebook, *Session options*):
1. *Accelerator* → **GPU T4 x2**.
2. *Internet* → **On** (hay que verificar el teléfono en la cuenta de Kaggle).

**Uso**
1. Ejecuta las celdas en orden (*Run All* sirve). La descarga de modelos tarda unos minutos.
2. La celda del túnel imprime `OLLAMA_BASE_URL=https://….trycloudflare.com`. Cópiala en el `.env` del backend y reinicia el backend.
3. **Deja la última celda ejecutándose**: mantiene viva la sesión. Si la detienes o cierras el notebook, el servidor se apaga.

La URL cambia en cada sesión. Cualquiera con la URL puede usar el modelo: no la publiques.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Instalar Ollama (la última versión: qwen3-vl necesita una reciente)
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Arrancar el servidor en segundo plano.
# Con 2 GPU de 16 GB caben cargados a la vez el chat y el modelo de visión (sin pausas al alternar).
import os, subprocess, time
env = dict(os.environ, OLLAMA_HOST='0.0.0.0:11434', OLLAMA_KEEP_ALIVE='24h', OLLAMA_MAX_LOADED_MODELS='2')
ollama = subprocess.Popen(['ollama', 'serve'], env=env, stdout=open('ollama.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(6)
!curl -s http://localhost:11434/api/version

In [ ]:
%%writefile Modelfile
# Mismo contenido que ollama/Modelfile del backend
FROM qwen2.5-coder:7b
PARAMETER temperature 0.3
PARAMETER top_p 0.9
PARAMETER repeat_penalty 1.05
PARAMETER num_ctx 8192
SYSTEM """
Eres Trazo, desarrollador senior y mentor de modelado UML 2.5 de la herramienta UML Studio.
Acompañas a un equipo que diseña diagramas de clases en una pizarra compartida y que después
genera un backend Spring Boot (JPA + PostgreSQL) a partir del diagrama. Hablas en español,
llamas a cada persona por su nombre, explicas el porqué de cada decisión de modelado y
propones mejoras concretas sin inventar elementos que no existan en el diagrama.
"""

In [ ]:
# Descargar los modelos y crear el modelo propio
!ollama pull qwen2.5-coder:7b
!ollama create trazo -f Modelfile
!ollama pull qwen3-vl:8b-instruct
!ollama list

In [ ]:
# Precargar ambos modelos en la GPU para que la primera petición del backend no espere
!curl -s http://localhost:11434/api/generate -d '{"model":"trazo","prompt":"hola","stream":false,"keep_alive":"24h"}' > /dev/null
!curl -s http://localhost:11434/api/generate -d '{"model":"qwen3-vl:8b-instruct","prompt":"hola","stream":false,"think":false,"keep_alive":"24h"}' > /dev/null
!ollama ps

In [ ]:
# Publicar el servidor con un túnel de Cloudflare (sin cuenta)
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
import re
tunnel = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://localhost:11434', '--http-host-header', 'localhost:11434'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tunnel.stdout:
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if match:
        print('OLLAMA_BASE_URL=' + match.group(0))
        break

In [ ]:
# Mantener viva la sesión: deja esta celda EJECUTÁNDOSE mientras uses la app.
# Cada minuto muestra la última línea del registro (cada petición del backend aparece aquí).
import time
while True:
    time.sleep(60)
    tail = open('ollama.log', errors='replace').read().strip().splitlines()[-1:]
    print(time.strftime('%H:%M'), tail[0][:150] if tail else '')